# PDP Diagnostic Validation — Fully Self-Contained NB1-Compatible Record

This notebook uses one separate, human-readable frozen workbook: `PDP_Diagnostic_Validation_FROZEN.xlsx`. The frozen Exp4 inputs and workbook are located using the same recursive, unambiguous file-discovery logic as NB1.

It contains:
1. NB1-compatible reconstruction from the frozen Exp4 inputs.
2. Empirical partition validation recomputed from the retained 3,090-block Baseline cohort.
3. Exact frozen pre-registration development artifacts stored in the companion workbook and hash-verified.
4. Exact original held-out blinded features and ground-truth key stored in the companion workbook and hash-verified.
5. Fresh application of the frozen classifier and fresh scoring of the 2,300 held-out datasets.

The held-out exercise is a **synthetic diagnostic-recovery validation**. It checks implementation/recovery under deliberately constructed statistical classes; it does not establish diagnostic validity for unknown physical disturbances.


## 1. Canonical NB1-compatible setup


In [2]:
import os
import ast
import pickle
import hashlib
import urllib.request
import json as _json
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import ks_2samp
from numpy.random import default_rng

warnings.filterwarnings("ignore")

STRUCTURAL_REVISION = "2026-08-04 structural gates 1-8; 2026-09-02 GitHub download fallback"

REQUIRED_INPUTS = {
    "blocks": "Frozen_Blocks_2026-02-10_195735*.csv",
    "sessions": "Frozen_Sessions_2026-02-10_195735*.csv",
    "participants": "Frozen_Participants_2026-02-10_195735*.csv",
    "audits": "Frozen_Audits_2026-02-10_195735*.csv",
    "raw_calls": "Frozen_Exp4_RawBlockBits_2026-07-26*.pkl",
    "diagnostic_workbook": "PDP_Diagnostic_Validation_FROZEN*.xlsx",
}

DOWNLOAD_BASE = "https://raw.githubusercontent.com/catboxer/Beyond-The-Mean/main/data"

def locate_required_inputs(filenames, search_root=None, download_base=DOWNLOAD_BASE):
    """Find one unambiguous local copy of every frozen input file, downloading
    it from the repo's data/ folder on GitHub if no local copy is found."""
    root = Path.cwd() if search_root is None else Path(search_root)
    data_dir = root / "data"
    resolved = {}
    problems = []
    for label, filename_pattern in filenames.items():
        # Common layouts are checked first (root, then its data/ subfolder).
        # Deliberately no recursive filesystem fallback: a broad rglob() from
        # root can sweep in unrelated stale copies of the same filename from
        # anywhere else in the filesystem (e.g. a mounted Google Drive with
        # old duplicate data), which then trips the DUPLICATED check below
        # before the download fallback ever gets a chance to run.
        candidates = []
        for parent in (root, data_dir):
            if parent.is_dir():
                candidates.extend(path.resolve() for path in parent.glob(filename_pattern) if path.is_file())
        candidates = sorted(set(candidates))
        if not candidates and download_base:
            exact_name = filename_pattern.replace("*", "")
            data_dir.mkdir(exist_ok=True)
            dest = data_dir / exact_name
            url = f"{download_base}/{exact_name}"
            print(f"  {label}: not found locally -- downloading {url}")
            try:
                urllib.request.urlretrieve(url, dest)
                candidates = [dest.resolve()]
            except Exception as exc:
                problems.append(f"MISSING: {filename_pattern} (download failed: {exc})")
                continue
        if len(candidates) == 1:
            resolved[label] = str(candidates[0])
        elif len(candidates) == 0:
            problems.append(f"MISSING: {filename_pattern}")
        else:
            locations = ", ".join(str(path) for path in candidates)
            problems.append(f"DUPLICATED accepted input for {label}: {locations}")
    if problems:
        expected = "\n".join(f"  - {pattern}" for pattern in filenames.values())
        details = "\n".join(problems)
        raise FileNotFoundError(
            "Frozen input-file check failed.\n"
            f"{details}\n\n"
            "Place exactly one copy of each required file in the notebook's working "
            "folder or its data/ subfolder:\n"
            f"{expected}\n"
            f"Current search root: {root.resolve()}"
        )
    return resolved

INPUT_PATHS = locate_required_inputs(REQUIRED_INPUTS)
FROZEN_BLOCKS_PATH = INPUT_PATHS["blocks"]
FROZEN_SESSIONS_PATH = INPUT_PATHS["sessions"]
FROZEN_PARTICIPANTS_PATH = INPUT_PATHS["participants"]
FROZEN_AUDITS_PATH = INPUT_PATHS["audits"]
FROZEN_RAW_BITS_PATH = INPUT_PATHS["raw_calls"]
PDPV_FROZEN_WORKBOOK_PATH = INPUT_PATHS["diagnostic_workbook"]

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_manifest = pd.DataFrame([
    {
        "label": label,
        "filename": Path(path).name,
        "bytes": Path(path).stat().st_size,
        "sha256": sha256_file(path),
    }
    for label, path in INPUT_PATHS.items()
])

# Verify the five pilot-dataset inputs (not the diagnostic workbook, which is
# checked separately below against its own hardcoded expected hash) against
# data/manifest.json. Downloads that manifest too if it isn't present yet,
# and fails loudly on any mismatch or missing entry.
_manifest_path = Path(INPUT_PATHS["blocks"]).parent / "manifest.json"
if not _manifest_path.is_file() and DOWNLOAD_BASE:
    print(f"  manifest.json: not found locally -- downloading {DOWNLOAD_BASE}/manifest.json")
    try:
        urllib.request.urlretrieve(f"{DOWNLOAD_BASE}/manifest.json", _manifest_path)
    except Exception as _exc:
        raise FileNotFoundError(
            f"data/manifest.json not found at {_manifest_path}, and downloading it "
            f"from {DOWNLOAD_BASE}/manifest.json also failed: {_exc}"
        )
if not _manifest_path.is_file():
    raise FileNotFoundError(
        f"data/manifest.json not found at {_manifest_path} -- run "
        "analysis/generate_manifest.py first."
    )
with open(_manifest_path) as _handle:
    _stored_manifest = _json.load(_handle)
_verification_problems = []
for _row in input_manifest.itertuples():
    if _row.label == "diagnostic_workbook":
        continue
    _stored = _stored_manifest["files"].get(_row.label)
    if _stored is None:
        _verification_problems.append(f"{_row.label}: no manifest.json entry")
    elif _stored["sha256"] != _row.sha256:
        _verification_problems.append(
            f"{_row.label} ({_row.filename}): hash mismatch -- "
            f"manifest={_stored['sha256'][:12]}... actual={_row.sha256[:12]}..."
        )
if _verification_problems:
    raise ValueError(
        "Frozen-input verification against manifest.json FAILED:\n"
        + "\n".join(_verification_problems)
    )
print(f"\n✓ FULL VERIFICATION PASSED -- 5 frozen pilot inputs match data/manifest.json "
      "(diagnostic_workbook checked separately below)")

print("Loading frozen data once...")
df_blocks_raw = pd.read_csv(FROZEN_BLOCKS_PATH)
df_sessions_raw = pd.read_csv(FROZEN_SESSIONS_PATH)
df_participants_raw = pd.read_csv(FROZEN_PARTICIPANTS_PATH)
df_audits_raw = pd.read_csv(FROZEN_AUDITS_PATH)

with open(FROZEN_RAW_BITS_PATH, "rb") as handle:
    raw_call_mapping = pickle.load(handle)

assert isinstance(raw_call_mapping, dict), "Raw-call pickle must be a session-keyed dictionary"
raw_call_rows = [
    (session_id, block_idx, bits301)
    for session_id, session_calls in raw_call_mapping.items()
    for block_idx, bits301 in session_calls
]
df_raw_calls = pd.DataFrame(raw_call_rows, columns=["session_id", "block_idx", "bits301"])
assert not df_raw_calls.duplicated(["session_id", "block_idx"]).any(), "Duplicate raw-call key"
valid_301 = df_raw_calls["bits301"].apply(
    lambda value: isinstance(value, str) and len(value) == 301 and set(value) <= {"0", "1"}
)
assert valid_301.all(), f"Invalid 301-bit strings: {(~valid_301).sum()}"
df_raw_calls["bit0"] = df_raw_calls["bits301"].str[0].astype("int8")
df_raw_calls["halfA"] = df_raw_calls["bits301"].str[1:151].apply(
    lambda value: np.fromiter((int(bit) for bit in value), dtype=np.int8, count=150)
)
df_raw_calls["halfB"] = df_raw_calls["bits301"].str[151:301].apply(
    lambda value: np.fromiter((int(bit) for bit in value), dtype=np.int8, count=150)
)
df_raw_calls["raw_sha256"] = df_raw_calls["bits301"].apply(
    lambda value: hashlib.sha256(value.encode("ascii")).hexdigest()
)

# Compatibility names: these objects are never mutated downstream.
df_blocks = df_blocks_raw.copy()
df_participants = df_participants_raw.copy()
df_audits = df_audits_raw.copy()

assert not df_sessions_raw["sessionId"].duplicated().any(), "Duplicate sessionId in Sessions"
assert not df_participants_raw["participant_id"].duplicated().any(), "Duplicate participant_id"
assert not df_blocks_raw.duplicated(["sessionId", "block_idx"]).any(), "Duplicate block key"

df_sessions = df_sessions_raw.rename(
    columns={"sessionId": "session_id", "agent_class": "condition"}
).copy()
df_audits_norm = df_audits_raw.rename(columns={"sessionId": "session_id"}).copy()

def parse_trial_data(value):
    if isinstance(value, dict):
        return value
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, dict) else None
    except (ValueError, SyntaxError, TypeError):
        return None

def hurstApprox(bits):
    """Single-scale R/S estimator matching the experiment's JS implementation."""
    if bits is None:
        return np.nan
    try:
        bits = np.asarray(bits, dtype=int)
        n = bits.size
        if n < 20:
            return np.nan
        x = np.where(bits != 0, 1.0, -1.0)
        mean = float(x.sum() / n)
        y = min_y = max_y = s2 = 0.0
        for value in x:
            deviation = value - mean
            y += deviation
            min_y = min(min_y, y)
            max_y = max(max_y, y)
            s2 += deviation * deviation
        spread = max_y - min_y
        scale = np.sqrt(s2 / n) if s2 > 0 else 1.0
        ratio = spread / scale if np.isfinite(scale) and scale != 0 else 1.0
        ratio = ratio if np.isfinite(ratio) and ratio != 0 else 1.0
        estimate = np.log(ratio) / np.log(n)
        estimate = estimate if np.isfinite(estimate) else 0.5
        return float(np.clip(estimate, 0.0, 1.0))
    except (TypeError, ValueError):
        return np.nan

# Normalize blocks without altering the raw table.
df_blocks_norm = df_blocks_raw.rename(columns={"sessionId": "session_id"}).copy()
df_blocks_norm["parsed"] = df_blocks_norm["trial_data"].apply(parse_trial_data)
for column, key in {
    "subject_bits": "subject_bits",
    "demon_bits": "demon_bits",
    "target_bit": "target_bit",
    "trial_count": "trial_count",
}.items():
    df_blocks_norm[column] = df_blocks_norm["parsed"].apply(
        lambda value, k=key: value.get(k) if value else None
    )

def target_aligned_rate(bits, target):
    """Fraction of bits equal to this row's target (not the fraction of ones)."""
    if target not in (0, 1) or not isinstance(bits, (list, tuple, np.ndarray)):
        return np.nan
    array = np.asarray(bits, dtype=np.int8)
    return float(np.mean(array == int(target)))

def one_fraction(bits):
    """Literal fraction of ones; a composition metric, never a hit rate."""
    if not isinstance(bits, (list, tuple, np.ndarray)):
        return np.nan
    return float(np.mean(np.asarray(bits, dtype=np.int8)))

df_blocks_norm["subject_hit_rate"] = df_blocks_norm["hits"] / 150.0
df_blocks_norm["pcs_hit_rate"] = df_blocks_norm["demon_hits"] / 150.0
df_blocks_norm["delta_hit"] = (
    df_blocks_norm["subject_hit_rate"] - df_blocks_norm["pcs_hit_rate"]
)

# Explicit composition variables prevent target-0 blocks from being scored as if
# the target were always 1.
df_blocks_norm["subject_one_fraction"] = df_blocks_norm["subject_bits"].apply(one_fraction)
df_blocks_norm["pcs_one_fraction"] = df_blocks_norm["demon_bits"].apply(one_fraction)
df_blocks_norm["subject_hit_rate_reconstructed"] = df_blocks_norm.apply(
    lambda row: target_aligned_rate(row["subject_bits"], row["target_bit"]), axis=1
)
df_blocks_norm["pcs_hit_rate_reconstructed"] = df_blocks_norm.apply(
    lambda row: target_aligned_rate(row["demon_bits"], row["target_bit"]), axis=1
)

# Compatibility aliases used by legacy downstream cells.
df_blocks_norm["rate_subject"] = df_blocks_norm["subject_hit_rate"]
df_blocks_norm["rate_pcs"] = df_blocks_norm["pcs_hit_rate"]
df_blocks_norm["rate_demon"] = df_blocks_norm["pcs_hit_rate"]
df_blocks_norm["delta_rate"] = df_blocks_norm["delta_hit"]
df_blocks_norm["subject_bit_length"] = df_blocks_norm["subject_bits"].apply(
    lambda value: len(value) if isinstance(value, (list, tuple, np.ndarray)) else np.nan
)
df_blocks_norm["demon_bit_length"] = df_blocks_norm["demon_bits"].apply(
    lambda value: len(value) if isinstance(value, (list, tuple, np.ndarray)) else np.nan
)

print("Computing single-scale R/S ordering scores...")
df_blocks_norm["hurst_subject"] = df_blocks_norm["subject_bits"].apply(hurstApprox)
df_blocks_norm["hurst_demon"] = df_blocks_norm["demon_bits"].apply(hurstApprox)
df_blocks_norm["hurst_pcs"] = df_blocks_norm["hurst_demon"]
df_blocks_norm["delta_hurst"] = (
    df_blocks_norm["hurst_subject"] - df_blocks_norm["hurst_pcs"]
)
df_blocks_norm["hurst_delta"] = df_blocks_norm["delta_hurst"]  # compatibility alias

# Reconcile raw pre-split calls with frozen blocks before analysis filtering.
df_raw_block_reconciliation = df_raw_calls.merge(
    df_blocks_norm[["session_id", "block_idx", "block_id", "qrng_hash", "subject_bits", "demon_bits", "target_bit", "hits", "demon_hits", "subject_hit_rate_reconstructed", "pcs_hit_rate_reconstructed"]],
    on=["session_id", "block_idx"], how="outer", validate="one_to_one", indicator="_raw_block_join"
)
raw_calls_without_block = df_raw_block_reconciliation.loc[
    df_raw_block_reconciliation["_raw_block_join"] == "left_only",
    ["session_id", "block_idx", "bits301"]
].copy()
blocks_without_raw_call = df_raw_block_reconciliation.loc[
    df_raw_block_reconciliation["_raw_block_join"] == "right_only",
    ["session_id", "block_idx", "block_id"]
].copy()

raw_matched = df_raw_block_reconciliation.loc[
    df_raw_block_reconciliation["_raw_block_join"] == "both"
].copy()
hash_match = raw_matched["raw_sha256"].eq(raw_matched["qrng_hash"])
assert hash_match.all(), f"Raw-call hash mismatches: {(~hash_match).sum()}"

def reconstructed_labels_match(row):
    expected_subject, expected_pcs = (
        (row["halfA"], row["halfB"]) if row["bit0"] == 1
        else (row["halfB"], row["halfA"])
    )
    return (
        np.array_equal(np.asarray(row["subject_bits"], dtype=np.int8), expected_subject)
        and np.array_equal(np.asarray(row["demon_bits"], dtype=np.int8), expected_pcs)
    )

raw_matched["labels_reconstruct"] = raw_matched.apply(reconstructed_labels_match, axis=1)
assert raw_matched["labels_reconstruct"].all(), (
    f"Raw-call label reconstruction mismatches: {(~raw_matched['labels_reconstruct']).sum()}"
)

valid_scoring = (
    raw_matched["target_bit"].isin([0, 1])
    & raw_matched["subject_hit_rate_reconstructed"].notna()
    & raw_matched["pcs_hit_rate_reconstructed"].notna()
)
assert valid_scoring.all(), f"Unscorable matched rows: {(~valid_scoring).sum()}"
assert np.allclose(
    raw_matched["subject_hit_rate_reconstructed"], raw_matched["hits"] / 150.0
), "Stored Subject hit counts disagree with target-aligned raw-bit scoring"
assert np.allclose(
    raw_matched["pcs_hit_rate_reconstructed"], raw_matched["demon_hits"] / 150.0
), "Stored PCS hit counts disagree with target-aligned raw-bit scoring"

# Validate the many-blocks-to-one-session join and preserve unmatched rows for QA.
df_joined_raw = df_blocks_norm.merge(
    df_sessions[["session_id", "condition", "participant_id", "createdAt"]],
    on="session_id", how="left", validate="many_to_one", indicator="_session_join"
)
assert len(df_joined_raw) == len(df_blocks_norm), "Session join changed block-row count"
unmatched_block_sessions = df_joined_raw.loc[
    df_joined_raw["_session_join"] == "left_only", ["session_id", "block_id", "block_idx"]
].copy()

# Participant metadata are not present for all system/AI identifiers. Preserve and
# report those unmatched identifiers rather than silently calling them participants.
participant_metadata = df_participants_raw.rename(
    columns={c: f"participant_{c}" for c in df_participants_raw.columns if c != "participant_id"}
)
df_joined_raw = df_joined_raw.merge(
    participant_metadata, on="participant_id", how="left", validate="many_to_one",
    indicator="_participant_join"
)
assert len(df_joined_raw) == len(df_blocks_norm), "Participant join changed block-row count"
unmatched_participant_sessions = df_joined_raw.loc[
    df_joined_raw["_participant_join"] == "left_only",
    ["session_id", "participant_id", "condition"]
].drop_duplicates().copy()

# Audits are one-to-many within session. Keep the row-level audit table separate and
# join only a one-row-per-session summary to blocks to avoid multiplying block rows.
df_audit_session = (
    df_audits_norm.groupby("session_id", as_index=False)
    .agg(
        audit_count=("audit_id", "size"),
        audit_failures=("is_random", lambda x: int((x == False).sum())),
        audit_nist_complete=("nist_all_pass", lambda x: int(x.notna().sum())),
        audit_first_timestamp=("qrng_timestamp", "min"),
        audit_last_timestamp=("qrng_timestamp", "max"),
    )
)
df_joined_raw = df_joined_raw.merge(
    df_audit_session, on="session_id", how="left", validate="many_to_one"
)
assert len(df_joined_raw) == len(df_blocks_norm), "Audit-summary join changed block-row count"

# Canonical analysis cohort: required outcomes/session metadata, then >=25 retained
# blocks per session. Every exclusion is recorded for the integrity audit.
required_mask = df_joined_raw[["rate_subject", "rate_demon", "condition"]].notna().all(axis=1)
df_required = df_joined_raw.loc[required_mask].copy()
required_session_sizes = df_required.groupby("session_id").size()
eligible_session_ids = required_session_sizes[required_session_sizes >= 25].index
eligible_mask = df_required["session_id"].isin(eligible_session_ids)
df = df_required.loc[eligible_mask].copy()

# Every retained analysis block must have an authenticated raw call. Raw-only calls
# are retained in the reconciliation ledger but cannot silently enter analyses.
retained_raw_check = df[["session_id", "block_idx"]].merge(
    df_raw_calls[["session_id", "block_idx"]],
    on=["session_id", "block_idx"], how="left", validate="one_to_one", indicator=True
)
assert retained_raw_check["_merge"].eq("both").all(), (
    f"Retained blocks lacking raw 301-bit calls: "
    f"{(~retained_raw_check['_merge'].eq('both')).sum()}"
)

valid_session_counts = (
    df.groupby(["participant_id", "condition"])["session_id"]
    .nunique().rename("session_count").reset_index()
)
df = df.merge(
    valid_session_counts, on=["participant_id", "condition"], how="left", validate="many_to_one"
)
df["session_count"] = df["session_count"].fillna(1).astype(int)

analysis_units = pd.DataFrame([
    {"object": "raw QRNG call", "key": "session_id + block_idx", "role": "physical source observation"},
    {"object": "paired block", "key": "session_id + block_idx", "role": "row-aligned Subject−PCS estimand"},
    {"object": "session", "key": "session_id", "role": "minimum resampling cluster for condition/session inference"},
    {"object": "Human participant", "key": "participant_id", "role": "higher cluster for repeated-Human sensitivity"},
])

def cluster_bootstrap_mean(data, value_col, cluster_col="session_id", n_boot=5000, seed=20260804):
    """Block-weighted mean with whole-cluster bootstrap uncertainty."""
    work = data[[cluster_col, value_col]].dropna()
    grouped = work.groupby(cluster_col)[value_col].agg(["sum", "count"])
    keys = grouped.index.to_numpy()
    rng_local = default_rng(seed)
    draws = np.empty(n_boot)
    for i in range(n_boot):
        sampled = rng_local.choice(keys, size=len(keys), replace=True)
        draws[i] = grouped.loc[sampled, "sum"].sum() / grouped.loc[sampled, "count"].sum()
    estimate = work[value_col].mean()
    lo, hi = np.quantile(draws, [0.025, 0.975])
    return {"estimate": estimate, "ci_low": lo, "ci_high": hi, "n_clusters": len(keys), "n_boot": n_boot}

def clustered_paired_signflip(data, value_col, cluster_col="session_id", n_perm=10000, seed=20260804):
    """Sign-flip whole session clusters; pairing is preserved inside each row."""
    work = data[[cluster_col, value_col]].dropna()
    grouped = work.groupby(cluster_col)[value_col].agg(["sum", "count"])
    observed = work[value_col].mean()
    rng_local = default_rng(seed)
    signs = rng_local.choice((-1.0, 1.0), size=(n_perm, len(grouped)))
    null = (signs * grouped["sum"].to_numpy()).sum(axis=1) / grouped["count"].sum()
    p_value = (np.count_nonzero(np.abs(null) >= abs(observed)) + 1) / (n_perm + 1)
    return {"estimate": observed, "p_value": p_value, "n_clusters": len(grouped), "n_perm": n_perm}

setup_exclusions = pd.DataFrame([
    {"stage": "raw blocks", "excluded_rows": 0, "remaining_rows": len(df_blocks_norm)},
    {"stage": "missing required outcome/session metadata", "excluded_rows": int((~required_mask).sum()), "remaining_rows": len(df_required)},
    {"stage": "sessions with fewer than 25 retained blocks", "excluded_rows": int((~eligible_mask).sum()), "remaining_rows": len(df)},
])

# Reusable canonical subsets. Downstream analyses should create local views and must
# not overwrite these names.
df_hurst = df.dropna(subset=["hurst_subject", "hurst_demon"]).copy()
df_valid_bits = df.loc[
    df["subject_bit_length"].eq(150) & df["demon_bit_length"].eq(150)
].copy()
df_repeat = df[df["session_count"] >= 5].copy()
df_repeat_hurst = df_hurst[df_hurst["session_count"] >= 5].copy()
df_single = df[df["session_count"] == 1].copy()
human_df = df[df["condition"] == "human"].copy()
ai_df = df[df["condition"] == "ai_agent"].copy()
base_df = df[df["condition"] == "baseline"].copy()
df_base = base_df.copy()  # compatibility alias; do not overwrite downstream
df_base_h = df_hurst[df_hurst["condition"] == "baseline"].copy()
base_hurst_df = df_base_h.copy()


df_raw_calls_with_session = df_raw_calls.merge(
    df_sessions[["session_id", "condition", "participant_id", "createdAt"]],
    on="session_id", how="left", validate="many_to_one"
)
baseline_raw_calls = df_raw_calls_with_session.loc[
    df_raw_calls_with_session["condition"].eq("baseline")
    & df_raw_calls_with_session["session_id"].isin(eligible_session_ids)
].copy()

h_1 = human_df[human_df["session_count"] == 1].copy()
h_2_4 = human_df[human_df["session_count"].between(2, 4)].copy()
h_5plus = human_df[human_df["session_count"] >= 5].copy()
n_human_total = human_df["participant_id"].nunique()

print("\nFrozen-input manifest")
print(input_manifest.to_string(index=False))
print("\nCanonical dataset ready")
print(setup_exclusions.to_string(index=False))
print("\nBlocks by condition:", df["condition"].value_counts().to_dict())
print("Sessions:", df["session_id"].nunique())
print("Human participants:", n_human_total)
print("Unmatched raw block rows retained for QA:", len(unmatched_block_sessions))
print("Raw calls / matched blocks:", len(df_raw_calls), len(raw_matched))
print("Raw calls without frozen block / blocks without raw call:", len(raw_calls_without_block), len(blocks_without_raw_call))
print("Baseline raw calls:", len(baseline_raw_calls))
print("Canonical paired columns: subject_hit_rate, pcs_hit_rate, delta_hit, hurst_subject, hurst_pcs, delta_hurst")
print("Objects: raw tables, normalized tables, df_joined_raw, df, df_hurst, df_valid_bits,")
print("         df_repeat, df_repeat_hurst, df_single, human_df, ai_df, base_df,")
print("         df_base, df_base_h, df_audit_session, df_raw_calls, baseline_raw_calls,")
print("         df_raw_block_reconciliation, setup_exclusions")


Loading frozen data once...
Computing single-scale R/S ordering scores...

Frozen-input manifest
              label                                  filename    bytes                                                           sha256
             blocks       Frozen_Blocks_2026-02-10_195735.csv 22044201 3328875750873c704abc47c872064c4878cb0f9d7f621a95dfdcc393b9f0b529
           sessions     Frozen_Sessions_2026-02-10_195735.csv   107732 6b54ae50ba65bcfaf2fe5a4f452765540dee2e1a3afd2da524418e5bc97f83b9
       participants Frozen_Participants_2026-02-10_195735.csv    55638 9debd57c983725c2bd770e961a8fc34d5d405c8624a6b734a3d5a60d9ef362d3
             audits       Frozen_Audits_2026-02-10_195735.csv  2366384 3e6779ae2ac009934dd9360a18ed8421c7900e0d07c05eeaacfe254253b1e1d8
          raw_calls   Frozen_Exp4_RawBlockBits_2026-07-26.pkl  3959136 6cf3248d3ef2ac34f3bf9d18faaba5aeb72fdd19b9c3d78c0b3ed81e868498f7
diagnostic_workbook     PDP_Diagnostic_Validation_FROZEN.xlsx   743192 dd45d49510526464

### Structural Integrity Gates 1–4

**Scope:** frozen-data provenance and joins; raw-call split/assignment reconstruction; target-aligned scoring; and explicit analysis units.

This cell is fail-loud. A failed assertion means later analysis must not run. Passing these checks establishes that the supplied frozen files were loaded consistently and the retained rows encode the stated architecture. It does not establish statistical independence, unbiasedness, or artifact cancellation.


In [3]:
# STRUCTURAL GATES 1–4 — fail-loud architecture checks
print("=" * 76)
print("STRUCTURAL GATES 1–4: DATA, RECONSTRUCTION, TARGET SCORING, UNITS")
print("=" * 76)

expected_counts = {"frozen_blocks": 11764, "retained_blocks": 11607, "excluded_rows": 157}
assert len(df_blocks_norm) == expected_counts["frozen_blocks"]
assert len(df) == expected_counts["retained_blocks"]
assert len(df_blocks_norm) - len(df) == expected_counts["excluded_rows"]
assert setup_exclusions["excluded_rows"].sum() == expected_counts["excluded_rows"]
assert len(unmatched_block_sessions) == 100
assert len(df_required) - len(df) == 57

# Join cardinality and retained raw coverage
assert not df_blocks_norm.duplicated(["session_id", "block_idx"]).any()
assert not df_sessions.duplicated("session_id").any()
assert not df_raw_calls.duplicated(["session_id", "block_idx"]).any()
assert len(df_joined_raw) == len(df_blocks_norm)
assert retained_raw_check["_merge"].eq("both").all()

# Raw call: bit 0 assigns labels; the two remaining halves are exactly 150 bits.
assert valid_301.all()
assert hash_match.all()
assert raw_matched["labels_reconstruct"].all()
assert raw_matched["halfA"].apply(len).eq(150).all()
assert raw_matched["halfB"].apply(len).eq(150).all()

# Target scoring: both target values must be represented and every stored count must
# equal raw-bit scoring against that row's actual target.
assert set(df["target_bit"].unique()) == {0, 1}
assert np.allclose(df["subject_hit_rate"], df["subject_hit_rate_reconstructed"])
assert np.allclose(df["pcs_hit_rate"], df["pcs_hit_rate_reconstructed"])
assert np.allclose(df["delta_hit"], df["subject_hit_rate"] - df["pcs_hit_rate"])
assert np.allclose(df_hurst["delta_hurst"], df_hurst["hurst_subject"] - df_hurst["hurst_pcs"])

integrity_report = pd.DataFrame([
    {"gate": "1 Frozen data and joins", "result": "verified", "detail": "11,764 raw block rows → 11,607 retained; 157 exclusions reconciled"},
    {"gate": "2 Stream reconstruction", "result": "verified", "detail": f"{len(raw_matched):,} matched calls; hashes and labels agree for every matched row"},
    {"gate": "3 Target alignment", "result": "verified", "detail": f"target 0={int((df.target_bit==0).sum()):,}; target 1={int((df.target_bit==1).sum()):,}; zero scoring mismatches"},
    {"gate": "4 Analysis units", "result": "defined", "detail": "block point estimands; session/participant clustering for inference"},
])
print(integrity_report.to_string(index=False))
print("\nExclusion ledger")
print(setup_exclusions.to_string(index=False))
print("\nAnalysis-unit registry")
print(analysis_units.to_string(index=False))
print("\nThese checks verify data encoding and reconstruction only; they do not establish statistical validity beyond those checks.")


STRUCTURAL GATES 1–4: DATA, RECONSTRUCTION, TARGET SCORING, UNITS
                   gate   result                                                              detail
1 Frozen data and joins verified  11,764 raw block rows → 11,607 retained; 157 exclusions reconciled
2 Stream reconstruction verified 11,664 matched calls; hashes and labels agree for every matched row
     3 Target alignment verified             target 0=5,732; target 1=5,875; zero scoring mismatches
       4 Analysis units  defined block point estimands; session/participant clustering for inference

Exclusion ledger
                                      stage  excluded_rows  remaining_rows
                                 raw blocks              0           11764
  missing required outcome/session metadata            100           11664
sessions with fewer than 25 retained blocks             57           11607

Analysis-unit registry
           object                    key                                               

## 2. Exact retained Baseline reconstruction


In [4]:
PDPV_base = baseline_raw_calls.merge(
    base_df[["session_id","block_idx","target_bit","subject_hit_rate","pcs_hit_rate",
             "delta_hit","hurst_subject","hurst_pcs","delta_hurst"]],
    on=["session_id","block_idx"], how="inner", validate="one_to_one"
).sort_values(["session_id","block_idx"]).reset_index(drop=True)

assert len(PDPV_base) == 3090
assert PDPV_base["session_id"].nunique() == 103

PDPV_A0 = np.stack(PDPV_base["halfA"].to_numpy()).astype(np.int8)
PDPV_B0 = np.stack(PDPV_base["halfB"].to_numpy()).astype(np.int8)
PDPV_assignment = PDPV_base["bit0"].to_numpy(dtype=np.int8)
PDPV_sessions = PDPV_base["session_id"].to_numpy()

PDPV_subject = np.where(PDPV_assignment[:,None] == 1, PDPV_A0, PDPV_B0)
PDPV_pcs = np.where(PDPV_assignment[:,None] == 1, PDPV_B0, PDPV_A0)

PDPV_hS = np.array([hurstApprox(x) for x in PDPV_subject], float)
PDPV_hP = np.array([hurstApprox(x) for x in PDPV_pcs], float)

assert np.allclose(PDPV_hS, PDPV_base["hurst_subject"])
assert np.allclose(PDPV_hP, PDPV_base["hurst_pcs"])
assert np.allclose(PDPV_hS-PDPV_hP, PDPV_base["delta_hurst"])

print(f"Baseline blocks: {len(PDPV_base):,}")
print(f"Baseline sessions: {PDPV_base['session_id'].nunique():,}")
print(f"Mean paired HRS: {(PDPV_hS-PDPV_hP).mean():+.6f}")
print("Exact row-level reconstruction: PASS")

Baseline blocks: 3,090
Baseline sessions: 103
Mean paired HRS: -0.001741
Exact row-level reconstruction: PASS


## 3. Empirical partition validation


In [5]:
PDPV_PART_BOOT = 5000
PDPV_PART_SEED = 20260824
PDPV_bits300 = np.concatenate([PDPV_A0, PDPV_B0], axis=1)

def PDPV_hrs_matrix(bits):
    return np.array([hurstApprox(row) for row in bits], float)

def PDPV_row_corr(A,B):
    A=np.asarray(A,float); B=np.asarray(B,float)
    Ac=A-A.mean(axis=1,keepdims=True); Bc=B-B.mean(axis=1,keepdims=True)
    num=(Ac*Bc).sum(axis=1)
    den=np.sqrt((Ac*Ac).sum(axis=1)*(Bc*Bc).sum(axis=1))
    return np.divide(num,den,out=np.zeros_like(num),where=den>0)

parts={"contiguous":(PDPV_bits300[:,:150],PDPV_bits300[:,150:])}
pairs=PDPV_bits300.reshape(len(PDPV_bits300),150,2)
parts["odd_even"]=(pairs[:,:,0],pairs[:,:,1])

A_pr=np.empty((len(PDPV_base),150),np.int8)
B_pr=np.empty((len(PDPV_base),150),np.int8)
for i,row in PDPV_base.reset_index(drop=True).iterrows():
    seed=int.from_bytes(hashlib.sha256(
        f"PDP-PARTITION-V1|{row['session_id']}|{row['block_idx']}".encode()
    ).digest()[:8],"big")
    rng=np.random.default_rng(seed)
    choice=rng.integers(0,2,size=150); rr=np.arange(150)
    A_pr[i]=pairs[i,rr,choice]; B_pr[i]=pairs[i,rr,1-choice]
parts["paired_prng"]=(A_pr,B_pr)

pos=np.arange(300); takeA=((pos//2)%2)==0
parts["2x2_interleave"]=(PDPV_bits300[:,takeA],PDPV_bits300[:,~takeA])

rows=[]
for method,(A,B) in parts.items():
    hA=PDPV_hrs_matrix(A); hB=PDPV_hrs_matrix(B)
    hS=np.where(PDPV_assignment==1,hA,hB)
    hP=np.where(PDPV_assignment==1,hB,hA)
    bitr=PDPV_row_corr(A,B)
    tmp=pd.DataFrame({"session_id":PDPV_sessions,"hA":hA,"hB":hB,
                      "delta":hS-hP,"bitr":bitr})
    for sid,g in tmp.groupby("session_id",sort=False):
        rows.append({"partition":method,"session_id":sid,
                     "mean_delta":g["delta"].mean(),
                     "mean_bit_r0":g["bitr"].mean(),
                     "hrs_r":np.corrcoef(g["hA"],g["hB"])[0,1]})

PDPV_partition_sessions=pd.DataFrame(rows)
rng=np.random.default_rng(PDPV_PART_SEED)
bout=[]
for method,g in PDPV_partition_sessions.groupby("partition"):
    for metric in ["mean_delta","mean_bit_r0","hrs_r"]:
        v=g[metric].dropna().to_numpy(float)
        draws=np.array([v[rng.integers(0,len(v),len(v))].mean()
                        for _ in range(PDPV_PART_BOOT)])
        bout.append({"partition":method,"metric":metric,"estimate":v.mean(),
                     "ci_low":np.quantile(draws,.025),
                     "ci_high":np.quantile(draws,.975),
                     "n_sessions":len(v)})
PDPV_partition_bootstrap=pd.DataFrame(bout)
display(PDPV_partition_bootstrap.round(6))

for _,g in PDPV_partition_bootstrap.groupby("metric"):
    assert ((g.ci_low<=0)&(g.ci_high>=0)).all()

print("\nCONCLUSION: no tested partition rule produced a resolved generic paired or")
print("contemporaneous relational offset in retained Baseline data.")

,partition,metric,estimate,ci_low,ci_high,n_sessions
0,2x2_interleave,mean_delta,-0.001581,-0.003706,0.000566,103
1,2x2_interleave,mean_bit_r0,0.002275,-0.000882,0.005215,103
2,2x2_interleave,hrs_r,0.006050,-0.026381,0.039718,103
3,contiguous,mean_delta,-0.001741,-0.003870,0.000323,103
4,contiguous,mean_bit_r0,0.002081,-0.000710,0.004898,103
5,contiguous,hrs_r,0.003962,-0.029035,0.037713,103
6,odd_even,mean_delta,-0.000034,-0.002006,0.001892,103
7,odd_even,mean_bit_r0,0.001775,-0.000808,0.004402,103
8,odd_even,hrs_r,-0.003167,-0.036215,0.031342,103
9,paired_prng,mean_delta,0.001088,-0.001247,0.003408,103



CONCLUSION: no tested partition rule produced a resolved generic paired or
contemporaneous relational offset in retained Baseline data.


## 4. Load exact embedded frozen development and held-out artifacts


In [6]:
# The frozen workbook path was resolved by the NB1-style recursive
# input locator in the main setup cell above.
PDPV_EXPECTED_WORKBOOK_SHA256 = "dd45d49510526464321d2a574e6a24531237fc15673a6adfe0c9eade7dbb5f2d"

PDPV_observed_sha256 = sha256_file(
    PDPV_FROZEN_WORKBOOK_PATH
)

assert PDPV_observed_sha256 == PDPV_EXPECTED_WORKBOOK_SHA256, (
    "FROZEN DIAGNOSTIC WORKBOOK HASH MISMATCH.\n"
    f"Expected: {PDPV_EXPECTED_WORKBOOK_SHA256}\n"
    f"Observed: {PDPV_observed_sha256}\n"
    f"File: {PDPV_FROZEN_WORKBOOK_PATH}\n"
    "Do not continue until the correct frozen workbook is supplied."
)

PDPV_SHEETS = {'PDP_dev_relational_only_results.csv': 'Relational_Development', 'PDP_dev_marginal_only_results.csv': 'Marginal_Development', 'PDP_dev_coupling_stability_results.csv': 'Coupling_Stability', 'PDP_Lag_Minus2_to_Plus2_Development_Results.csv': 'Lag_Development', 'PDP_dev_blind_classifier_development_results.csv': 'Classifier_Development', 'PDP_HeldOut_Features_BLINDED.csv': 'HeldOut_Features', 'PDP_HeldOut_GroundTruth_KEY.csv': 'HeldOut_Key'}

def PDPV_frozen_sheet(original_csv_name):
    sheet_name = PDPV_SHEETS[original_csv_name]
    return pd.read_excel(
        PDPV_FROZEN_WORKBOOK_PATH,
        sheet_name=sheet_name
    )

print("Frozen diagnostic workbook: HASH VERIFIED")
print("SHA-256:", PDPV_observed_sha256)
print("Workbook:", Path(PDPV_FROZEN_WORKBOOK_PATH).resolve())


Frozen diagnostic workbook: HASH VERIFIED
SHA-256: dd45d49510526464321d2a574e6a24531237fc15673a6adfe0c9eade7dbb5f2d
Workbook: /content/drive/MyDrive/QART_Project/PDP_Diagnostic_Validation_FROZEN.xlsx


### 4.1 Relational-only development


In [7]:
PDPV_rel_dev=PDPV_frozen_sheet("PDP_dev_relational_only_results.csv")
PDPV_rel_summary=PDPV_rel_dev.groupby("rho_target",as_index=False).agg(
    mean_global_r=("global_r","mean"),
    mean_session_r=("mean_session_r","mean"),
    mean_delta=("mean_delta","mean"))
display(PDPV_rel_summary.round(6))
assert np.allclose(PDPV_rel_summary.mean_delta,PDPV_rel_summary.mean_delta.iloc[0])


,rho_target,mean_global_r,mean_session_r,mean_delta
0,0.00,-0.000855,-0.003513,-0.001741
1,0.25,0.232698,0.242529,-0.001741
2,0.50,0.468328,0.489566,-0.001741
3,0.75,0.697994,0.731317,-0.001741


### 4.2 Marginal-only development


In [8]:
PDPV_marg_dev=PDPV_frozen_sheet("PDP_dev_marginal_only_results.csv")
PDPV_marg_summary=PDPV_marg_dev.groupby("q",as_index=False).agg(
    mean_subject_HRS=("mean_hS","mean"),
    mean_pcs_HRS=("mean_hP","mean"),
    mean_delta=("mean_delta","mean"),
    mean_global_r=("global_r","mean"))
display(PDPV_marg_summary.round(6))


,q,mean_subject_HRS,mean_pcs_HRS,mean_delta,mean_global_r
0,0.00,0.527148,0.528889,-0.001741,0.000666
1,0.05,0.536281,0.528889,0.007393,0.000762
2,0.10,0.545515,0.528889,0.016627,0.000515
3,0.20,0.564060,0.528889,0.035171,0.002967


### 4.3 Lag localization development


In [9]:
PDPV_lag_dev=PDPV_frozen_sheet("PDP_Lag_Minus2_to_Plus2_Development_Results.csv")
PDPV_lag_summary=PDPV_lag_dev.groupby(["true_lag","rho"],as_index=False).agg(
    n=("correct","size"), correct=("correct","sum"), accuracy=("correct","mean"),
    mean_true_r=("true_r","mean"),
    mean_best_competing_r=("best_competing_r","mean"),
    mean_margin=("localization_margin","mean"),
    min_margin=("localization_margin","min"))
display(PDPV_lag_summary.round(6))
print(f"Localization accuracy: {int(PDPV_lag_dev.correct.sum()):,}/{len(PDPV_lag_dev):,}")
assert PDPV_lag_dev.correct.all()


,true_lag,rho,n,correct,accuracy,mean_true_r,mean_best_competing_r,mean_margin,min_margin
0,-2,0.25,100,100,1.0,0.243890,0.010426,0.233464,0.182364
1,-2,0.50,100,100,1.0,0.488446,0.005673,0.482773,0.433657
2,-2,0.75,100,100,1.0,0.731078,-0.000110,0.731188,0.693421
3,-1,0.25,100,100,1.0,0.245954,0.007825,0.238129,0.178226
4,-1,0.50,100,100,1.0,0.488727,-0.002562,0.491290,0.447413
5,-1,0.75,100,100,1.0,0.731326,-0.014952,0.746278,0.713428
6,0,0.25,100,100,1.0,0.243237,0.007528,0.235710,0.173602
7,0,0.50,100,100,1.0,0.491291,-0.003929,0.495220,0.431915
8,0,0.75,100,100,1.0,0.733413,-0.015842,0.749255,0.725992
9,1,0.25,100,100,1.0,0.244195,0.008278,0.235916,0.176826


Localization accuracy: 1,500/1,500


### 4.4 Coupling-stability development


In [10]:
PDPV_stability_dev=PDPV_frozen_sheet("PDP_dev_coupling_stability_results.csv")
PDPV_stability_dev["kind_display"]=PDPV_stability_dev["kind"].fillna("null")
PDPV_stability_summary=PDPV_stability_dev.groupby("kind_display",as_index=False).agg(
    n=("rep","size"), mean_local_r=("mean_local_r","mean"),
    mean_var_local_r=("var_local_r","mean"),
    mean_session_global_r=("mean_session_global_r","mean"))
display(PDPV_stability_summary.round(6))


,kind_display,n,mean_local_r,mean_var_local_r,mean_session_global_r
0,instability,100,-0.000065,0.827563,0.013377
1,null,100,0.000192,0.249185,0.004689


### 4.5 Development classifier record


In [11]:
PDPV_dev=PDPV_frozen_sheet("PDP_dev_blind_classifier_development_results.csv")
PDPV_dev["true_display"]=PDPV_dev["true_class"].fillna("null")
PDPV_dev["pred_display"]=PDPV_dev["predicted"].fillna("null")
PDPV_dev["correct"]=PDPV_dev.true_display.eq(PDPV_dev.pred_display)
display(pd.crosstab(PDPV_dev.true_display,PDPV_dev.pred_display))
print(f"Development classification: {int(PDPV_dev.correct.sum())}/{len(PDPV_dev)}")
print("Development only; thresholds were selected from development signatures.")


pred_display,instability,lag_minus1,lag_plus1,marginal_only,null,relational_zero
true_display,,,,,,
instability,50,0,0,0,0,0
lag_minus1,0,50,0,0,0,0
lag_plus1,0,0,50,0,0,0
marginal_only,0,0,0,50,0,0
null,0,0,0,0,50,0
relational_zero,0,0,0,0,0,50


Development classification: 300/300
Development only; thresholds were selected from development signatures.


## 5. Preregistered held-out synthetic diagnostic-recovery validation

Registration DOI: **10.17605/OSF.IO/2KRZ3**

The exact original blinded feature table and separate ground-truth key are embedded and hash-verified below.


In [12]:
PDPV_features=PDPV_frozen_sheet("PDP_HeldOut_Features_BLINDED.csv")
PDPV_key=PDPV_frozen_sheet("PDP_HeldOut_GroundTruth_KEY.csv")
assert len(PDPV_features)==2300 and len(PDPV_key)==2300
assert PDPV_features.dataset_id.is_unique and PDPV_key.dataset_id.is_unique
print("Held-out datasets:",len(PDPV_features))
print("Feature table and key: HASH VERIFIED")


Held-out datasets: 2300
Feature table and key: HASH VERIFIED


### 5.1 Apply frozen classifier to blinded features only


In [13]:
PDPV_pred=PDPV_features[["dataset_id"]].copy()
PDPV_pred["marginal_flag"]=(PDPV_features.delta_shift_from_unperturbed>0.004)&(
    PDPV_features.zero_r_global.abs()<=0.15)
PDPV_pred["relational_zero_flag"]=PDPV_features.zero_r_global.abs()>0.15
lagcols=["lag_-2","lag_-1","lag_+0","lag_+1","lag_+2"]
PDPV_pred["predicted_lag"]=PDPV_features[lagcols].to_numpy().argmax(axis=1)-2
PDPV_pred["instability_flag"]=(PDPV_features.local_var_r_W10>0.50)&(
    PDPV_features.local_mean_r_W10.abs()<0.05)
print("Predictions generated without reading the condition key:",len(PDPV_pred))

Predictions generated without reading the condition key: 2300


### 5.2 Unblind and score applicable preregistered criteria


In [14]:
PDPV_scored=PDPV_key.merge(PDPV_pred,on="dataset_id",validate="one_to_one")
PDPV_scored["true_display"]=PDPV_scored["true_class"].fillna("null")

def criterion(r):
    c=r.true_display
    if c=="marginal_only": return bool(r.marginal_flag)
    if c=="relational_zero": return bool(r.relational_zero_flag)
    if c=="lagged_relational": return int(r.predicted_lag)==int(r.injected_lag)
    if c=="instability": return bool(r.instability_flag)
    if c=="null": return not (bool(r.marginal_flag) or bool(r.relational_zero_flag) or bool(r.instability_flag))
    raise ValueError(c)

PDPV_scored["criterion_met"]=PDPV_scored.apply(criterion,axis=1)
PDPV_class_summary=PDPV_scored.groupby("true_display",as_index=False).agg(
    n=("criterion_met","size"), correct=("criterion_met","sum"),
    accuracy=("criterion_met","mean"),
    marginal_flag_rate=("marginal_flag","mean"),
    relational_zero_flag_rate=("relational_zero_flag","mean"),
    instability_flag_rate=("instability_flag","mean"))
display(PDPV_class_summary)
print(f"Overall applicable criterion success: {int(PDPV_scored.criterion_met.sum()):,}/{len(PDPV_scored):,}")

,true_display,n,correct,accuracy,marginal_flag_rate,relational_zero_flag_rate,instability_flag_rate
0,instability,100,100,1.0,0.0,0.0,1.0
1,lagged_relational,1500,1500,1.0,0.0,0.2,0.0
2,marginal_only,300,300,1.0,1.0,0.0,0.0
3,null,100,100,1.0,0.0,0.0,0.0
4,relational_zero,300,300,1.0,0.0,1.0,0.0


Overall applicable criterion success: 2,300/2,300


### 5.3 Lag confusion matrix


In [15]:
PDPV_lagged=PDPV_scored[PDPV_scored.true_display=="lagged_relational"].copy()
PDPV_cm=pd.crosstab(PDPV_lagged.injected_lag.astype(int),PDPV_lagged.predicted_lag.astype(int))
display(PDPV_cm)
ok=(PDPV_lagged.injected_lag.astype(int)==PDPV_lagged.predicted_lag.astype(int)).sum()
print(f"Lag localization accuracy: {ok:,}/{len(PDPV_lagged):,}")

predicted_lag,-2,-1,0,1,2
injected_lag,,,,,
-2,300,0,0,0,0
-1,0,300,0,0,0
0,0,0,300,0,0
1,0,0,0,300,0
2,0,0,0,0,300


Lag localization accuracy: 1,500/1,500


### 5.4 Null controls


In [16]:
PDPV_null=PDPV_scored[PDPV_scored.true_display=="null"]
print("Null controls:",len(PDPV_null))
print("Marginal flags:",int(PDPV_null.marginal_flag.sum()))
print("Zero-lag relational flags:",int(PDPV_null.relational_zero_flag.sum()))
print("Instability flags:",int(PDPV_null.instability_flag.sum()))
print("\nNo no-lag claim is made because the preregistration did not define a no-lag threshold.")

Null controls: 100
Marginal flags: 0
Zero-lag relational flags: 0
Instability flags: 0

No no-lag claim is made because the preregistration did not define a no-lag threshold.


## 6. Bounded conclusion

The held-out exercise showed that the frozen implementation recovered all applicable targeted properties in the 2,300 deliberately constructed synthetic datasets.

It is an **implementation and synthetic-recovery check**, not independent validation that PDP can diagnose unknown physical-system disturbances.
